# 12｜设备工厂、机器狗与自定义传感器

本课全部使用 Mock，不依赖厂商 SDK。

![协同与智能能力](assets/advanced.svg)


## 同一套 Plan 控制机器狗

设备类别不同，Plan 会返回对应的强类型能力。机器狗使用 `stand / move / sit`，无人机使用 `takeoff / land`。


In [ ]:
from fleetdock import DeviceFactory, DeviceMetadata, Plan, RelativeMove2D

with (
    DeviceFactory.create()
    .mock_robot_dog()
    .metadata(DeviceMetadata(device_id="teaching-dog", name="教学机器狗"))
    .build()
) as dog:
    print(dog.connect())
    result = (
        Plan(dog)
        .stand()
        .move(RelativeMove2D(forward_m=2.0))
        .sit()
        .run()
    )
    print(result)


## 注册自定义传感器

扩展传感器仍然走标准 `SensorHub`。应用只定义传感器类型和样本字段，订阅、分发和生命周期由 SDK 管理。


In [ ]:
from fleetdock import (
    ExtensionSensorKind,
    GenericSensorSample,
    MockSensorBackend,
    SensorDescriptor,
)
import time

gas_kind = ExtensionSensorKind("custom.inspection.gas")
backend = MockSensorBackend([
    SensorDescriptor("gas-front", gas_kind),
])
drone = DeviceFactory.create().mock_drone().build()
drone.sensors.register_backend(backend)

unsubscribe = drone.sensors.subscribe(
    gas_kind,
    lambda sample: print("气体浓度：", sample.values["ppm"], "ppm"),
)
backend.publish(
    GenericSensorSample(
        sensor_id="gas-front",
        kind=gas_kind,
        sequence=1,
        values={"ppm": 12.5},
    )
)
time.sleep(0.1)

unsubscribe()
drone.close()


## 重点

`DeviceFactory` 隔离产品创建，`Plan` 隔离任务编排，`SensorHub` 隔离数据流。扩展设备或传感器时，应用仍保留统一写法。
